In [1]:
# Cell 1: imports & device

import os
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

from sklearn.preprocessing import StandardScaler

# Your own indicators
from technicals.indicators import BollingerBands, ATR, RSI, MACD

print("Torch:", torch.__version__)
print("Lightning:", pl.__version__)
print("CUDA available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
device


Torch: 2.5.1+cu121
Lightning: 2.5.6
CUDA available: True


'cuda'

In [2]:
# Cell 2: load M5 data from Parquet

DATA_FILE = Path(r"C:\Users\admin\Desktop\Forex_algo\code\data\EUR_USD_M5.parquet")

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Missing file: {DATA_FILE}")

df = pd.read_parquet(DATA_FILE)
df = df.sort_values("time").reset_index(drop=True)

print(df.head())
print(df.dtypes)
print("Rows:", len(df))


                       time  volume    mid_o    mid_h    mid_l    mid_c  \
0 2016-01-07 00:00:00+00:00      74  1.07764  1.07811  1.07759  1.07786   
1 2016-01-07 00:05:00+00:00      98  1.07788  1.07818  1.07764  1.07810   
2 2016-01-07 00:10:00+00:00      28  1.07812  1.07832  1.07812  1.07828   
3 2016-01-07 00:15:00+00:00      40  1.07824  1.07830  1.07798  1.07798   
4 2016-01-07 00:20:00+00:00      53  1.07796  1.07799  1.07776  1.07790   

     bid_o    bid_h    bid_l    bid_c    ask_o    ask_h    ask_l    ask_c  
0  1.07757  1.07802  1.07750  1.07777  1.07772  1.07820  1.07768  1.07795  
1  1.07779  1.07811  1.07755  1.07802  1.07798  1.07827  1.07772  1.07819  
2  1.07803  1.07823  1.07803  1.07819  1.07822  1.07840  1.07822  1.07837  
3  1.07815  1.07821  1.07790  1.07790  1.07834  1.07838  1.07807  1.07807  
4  1.07787  1.07789  1.07768  1.07782  1.07804  1.07809  1.07784  1.07797  
time      datetime64[ns, UTC]
volume                  int64
mid_o                 float64
mid

In [4]:
# Cell 3: features + target (log-return)

df_feat = df.copy()

# --- time handling ---
df_feat["time"] = pd.to_datetime(df_feat["time"])
df_feat = df_feat.sort_values("time").reset_index(drop=True)

# --- indicators using your functions ---
df_feat = RSI(df_feat, n=14)               # RSI_14
df_feat = MACD(df_feat)                    # MACD, SIGNAL, HIST
df_feat = ATR(df_feat, n=14)               # ATR_14
df_feat = BollingerBands(df_feat, n=20, s=2)  # BB_MA, BB_UP, BB_LW

# --- log return ---
df_feat["log_ret"] = np.log(df_feat["mid_c"] / df_feat["mid_c"].shift(1))

# --- target: next-candle log return ---
df_feat["target"] = df_feat["log_ret"].shift(-1)

# --- simple extra past-returns ---
df_feat["ret_3"] = df_feat["mid_c"].pct_change(3)
df_feat["ret_6"] = df_feat["mid_c"].pct_change(6)

# --- time features ---
df_feat["hour"] = df_feat["time"].dt.hour
df_feat["minute"] = df_feat["time"].dt.minute
df_feat["day_of_week"] = df_feat["time"].dt.dayofweek

# --- drop NaNs from indicators & target ---
df_feat = df_feat.dropna().reset_index(drop=True)

print(df_feat[["time", "mid_c", "log_ret", "target", "RSI_14", "MACD", "ATR_14", "BB_MA"]].head())
print("Rows after feature engineering:", len(df_feat))


                       time    mid_c   log_ret    target     RSI_14      MACD  \
0 2016-01-07 02:45:00+00:00  1.08146 -0.000018  0.000018  69.310433  0.000572   
1 2016-01-07 02:50:00+00:00  1.08148  0.000018  0.000037  69.461247  0.000587   
2 2016-01-07 02:55:00+00:00  1.08152  0.000037  0.000009  69.781094  0.000595   
3 2016-01-07 03:00:00+00:00  1.08153  0.000009  0.000213  69.866065  0.000595   
4 2016-01-07 03:05:00+00:00  1.08176  0.000213  0.000148  71.828165  0.000606   

     ATR_14     BB_MA  
0  0.000635  1.080179  
1  0.000604  1.080366  
2  0.000510  1.080527  
3  0.000453  1.080650  
4  0.000420  1.080756  
Rows after feature engineering: 735911


In [5]:
# Cell 4: direction labels + features

# Binary direction: up (1) / down (0)
df_feat["dir_label"] = (df_feat["target"] > 0).astype(int)

FEATURE_COLS = [
    "mid_o", "mid_h", "mid_l", "mid_c",
    "volume",
    "RSI_14", "MACD", "SIGNAL", "HIST",
    "ATR_14", "BB_MA", "BB_UP", "BB_LW",
    "log_ret", "ret_3", "ret_6",
    "hour", "minute", "day_of_week",
]

TARGET_COL = "target"
DIR_COL    = "dir_label"

X = df_feat[FEATURE_COLS].astype(np.float32).values
y_reg = df_feat[TARGET_COL].astype(np.float32).values  # regression
y_dir = df_feat[DIR_COL].astype(np.int64).values       # classification

n = len(df_feat)
train_end = int(0.7 * n)
val_end   = int(0.85 * n)

X_train, y_reg_train, y_dir_train = X[:train_end], y_reg[:train_end], y_dir[:train_end]
X_val,   y_reg_val,   y_dir_val   = X[train_end:val_end], y_reg[train_end:val_end], y_dir[train_end:val_end]
X_test,  y_reg_test,  y_dir_test  = X[val_end:], y_reg[val_end:], y_dir[val_end:]

# Scale features (NOT the target)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("Train/Val/Test sizes:", X_train.shape[0], X_val.shape[0], X_test.shape[0])


Train/Val/Test sizes: 515137 110387 110387


In [12]:
# Cell 5: Dataset & DataLoader with long sequences

SEQ_LEN = 256  # 256 * 5min ≈ 21.3 hours of history

class ForexSeqDataset(Dataset):
    def __init__(self, X, y_reg, y_dir, seq_len):
        self.X = X
        self.y_reg = y_reg
        self.y_dir = y_dir
        self.seq_len = seq_len

        # valid last indices: from seq_len .. len(X)-1
        self.indices = np.arange(seq_len, len(X))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]
        x_seq = self.X[t - self.seq_len:t]     # [seq_len, d_in]
        y_r = self.y_reg[t]                    # scalar
        y_d = self.y_dir[t]                    # 0 or 1

        x_seq = torch.from_numpy(x_seq).float()
        y_r = torch.tensor(y_r, dtype=torch.float32)
        y_d = torch.tensor(y_d, dtype=torch.long)

        return x_seq, (y_r, y_d)

train_ds = ForexSeqDataset(X_train_scaled, y_reg_train, y_dir_train, SEQ_LEN)
val_ds   = ForexSeqDataset(X_val_scaled,   y_reg_val,   y_dir_val,   SEQ_LEN)
test_ds  = ForexSeqDataset(X_test_scaled,  y_reg_test,  y_dir_test,  SEQ_LEN)

BATCH_SIZE = 1280  # your GPU can handle this; adjust if OOM

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

len(train_ds), len(val_ds), len(test_ds)


(514881, 110131, 110131)

In [13]:
# Cell 6: Transformer model with dual heads

class ForexTransformerV2(pl.LightningModule):
    def __init__(
        self,
        d_in: int,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 4,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
        lr: float = 3e-4,
        dir_loss_weight: float = 0.5,
    ):
        super().__init__()

        self.save_hyperparameters()

        self.input_proj = nn.Linear(d_in, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,   # [B, T, D]
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # regression head (log-return)
        self.head_reg = nn.Linear(d_model, 1)
        # direction head (UP/DOWN)
        self.head_dir = nn.Linear(d_model, 2)

        self.mse_loss = nn.MSELoss()
        self.ce_loss  = nn.CrossEntropyLoss()

    def forward(self, x):
        # x: [B, T, d_in]
        x = self.input_proj(x)         # [B, T, d_model]
        h = self.encoder(x)            # [B, T, d_model]
        h_last = h[:, -1, :]           # [B, d_model]

        out_reg = self.head_reg(h_last).squeeze(-1)  # [B]
        out_dir = self.head_dir(h_last)              # [B, 2] logits

        return out_reg, out_dir

    def training_step(self, batch, batch_idx):
        x, (y_reg, y_dir) = batch
        x = x.to(self.device)
        y_reg = y_reg.to(self.device)
        y_dir = y_dir.to(self.device)

        pred_reg, pred_dir = self(x)

        loss_reg = self.mse_loss(pred_reg, y_reg)
        loss_dir = self.ce_loss(pred_dir, y_dir)
        loss = loss_reg + self.hparams.dir_loss_weight * loss_dir

        with torch.no_grad():
            mae = torch.mean(torch.abs(pred_reg - y_reg))
            pred_dir_labels = torch.argmax(pred_dir, dim=1)
            acc_dir = torch.mean((pred_dir_labels == y_dir).float())

        self.log("train_loss", loss, prog_bar=True, on_epoch=True)
        self.log("train_loss_reg", loss_reg, prog_bar=False, on_epoch=True)
        self.log("train_loss_dir", loss_dir, prog_bar=False, on_epoch=True)
        self.log("train_mae_logret", mae, prog_bar=True, on_epoch=True)
        self.log("train_dir_acc", acc_dir, prog_bar=True, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, (y_reg, y_dir) = batch
        x = x.to(self.device)
        y_reg = y_reg.to(self.device)
        y_dir = y_dir.to(self.device)

        pred_reg, pred_dir = self(x)

        loss_reg = self.mse_loss(pred_reg, y_reg)
        loss_dir = self.ce_loss(pred_dir, y_dir)
        loss = loss_reg + self.hparams.dir_loss_weight * loss_dir

        mae = torch.mean(torch.abs(pred_reg - y_reg))
        pred_dir_labels = torch.argmax(pred_dir, dim=1)
        acc_dir = torch.mean((pred_dir_labels == y_dir).float())

        self.log("val_loss", loss, prog_bar=True, on_epoch=True)
        self.log("val_loss_reg", loss_reg, prog_bar=False, on_epoch=True)
        self.log("val_loss_dir", loss_dir, prog_bar=False, on_epoch=True)
        self.log("val_mae_logret", mae, prog_bar=True, on_epoch=True)
        self.log("val_dir_acc", acc_dir, prog_bar=True, on_epoch=True)

        return {"val_loss": loss, "val_mae_logret": mae, "val_dir_acc": acc_dir}

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=50, eta_min=1e-5
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
                "monitor": "val_loss",
            },
        }

d_in = len(FEATURE_COLS)
model = ForexTransformerV2(d_in=d_in)
model.to(device)
model


ForexTransformerV2(
  (input_proj): Linear(in_features=19, out_features=128, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (head_reg): Linear(in_features=128, out_features=1, bias=True)
  (head_dir): Linear(in_features=128, out_features=2, bias=True)
  (mse_loss): MSELoss()
  (ce_loss): CrossEntropyLoss()
)

In [14]:
# Cell 7: training with GPU + callbacks

checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="eurusd_m5_transformer_v2-{epoch:02d}-{val_loss:.6f}",
)

earlystop_cb = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    min_delta=1e-5,
)

accelerator = "gpu" if torch.cuda.is_available() else "cpu"
devices = 1

trainer = pl.Trainer(
    max_epochs=60,
    accelerator=accelerator,
    devices=devices,
    gradient_clip_val=0.5,
    precision="16-mixed" if accelerator == "gpu" else "32",
    callbacks=[checkpoint_cb, earlystop_cb],
    log_every_n_steps=50,
)

trainer.fit(model, train_loader, val_loader)

print("Best checkpoint:", checkpoint_cb.best_model_path)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name       | Type               | Params | Mode 
----------------------------------------------------------
0 | input_proj | Linear             | 2.6 K  | train
1 | encoder    | TransformerEncoder | 529 K  | train
2 | head_reg   | Linear             | 129    | train
3 | head_dir   | Linear             | 258    | train
4 | mse_loss   | MSELoss            | 0      | train
5 | ce_loss    | CrossEntropyLoss   | 0      | train
----------------------------------------------------------
532 K     Trainable params
0         Non-trainable params
532 K     Total params
2.131     Total estimated model params size (MB)
47        Modules in train mode
0         Modules in eval mode


Epoch 16: 100%|██████████| 403/403 [02:59<00:00,  2.24it/s, v_num=36, train_loss_step=0.344, train_mae_logret_step=0.00238, train_dir_acc_step=0.558, val_loss=0.346, val_mae_logret=0.000659, val_dir_acc=0.516, train_loss_epoch=0.346, train_mae_logret_epoch=0.00231, train_dir_acc_epoch=0.521]
Best checkpoint: c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_36\checkpoints\eurusd_m5_transformer_v2-epoch=06-val_loss=0.346188.ckpt
